In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!pip install -q langchain-text-splitters

In [3]:
import os
import re
import torch
import gc
import pandas as pd
import numpy as np
import json
import pickle
from tqdm import tqdm
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Extract text books

In [1]:
# ====================  TEXT EXTRACTION ====================
def extract_text_from_htm(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            html_content = f.read()

        soup = BeautifulSoup(html_content, 'html.parser')
        raw_text = soup.get_text(separator='\n')

        lines = [line.strip() for line in raw_text.split('\n') if line.strip()]
        text = '\n'.join(lines)
        clean_text = re.sub(r'\n\s*\n+', '\n\n', text).strip()
        return clean_text
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return ""

# load chunks and embeddings

In [5]:
import os
import pickle
import random
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

# ===================== CONFIG =====================
save_folder      = "/content/drive/MyDrive/Book_Embeddings_100/jinaa5_nano"
n_samples        = 1000
deletion_ratio   = 0.20
random_seed      = 42

random.seed(random_seed)
np.random.seed(random_seed)

# ===================== LOAD ALL SAVED EMBEDDINGS =====================
all_chunks_text = []
all_embeddings  = []
all_metadata    = []

pkl_files = [f for f in os.listdir(save_folder) if f.endswith('.pkl')]
print(f"Found {len(pkl_files)} embedding files")

for pkl_file in tqdm(pkl_files, desc="Loading embeddings"):
    file_path = os.path.join(save_folder, pkl_file)
    with open(file_path, "rb") as f:
        data = pickle.load(f)

    chunks     = data["chunks"]
    embeddings = data["embeddings"]
    book_title = data["book_title"]
    file_name  = data["file_name"]

    for i, (chunk, emb) in enumerate(zip(chunks, embeddings)):
        all_chunks_text.append(chunk)
        all_embeddings.append(emb)
        all_metadata.append({
            "file_name": file_name,
            "book_title": book_title,
            "chunk_id": i
        })

all_embeddings = np.array(all_embeddings).astype('float32')

print(f"""
============== LOADED DATA (jina-v5-nano) ==============
  Total books:      {len(pkl_files)}
  Total chunks:     {len(all_chunks_text)}
  Embedding dim:    {all_embeddings.shape[1]}
  Matrix size:      {all_embeddings.nbytes/1024**2:.2f} MB
=========================================================
""")

# ===================== SAMPLE 1000 RANDOM CHUNKS =====================
total_chunks = len(all_chunks_text)
sample_indices = random.sample(range(total_chunks), min(n_samples, total_chunks))

print(f"Sampled {len(sample_indices)} chunks out of {total_chunks} total")

sampled_chunks     = [all_chunks_text[i] for i in sample_indices]
sampled_metadata   = [all_metadata[i] for i in sample_indices]
sampled_embeddings = all_embeddings[sample_indices]

# ===================== DELETE 20% OF TOKENS RANDOMLY =====================
def delete_random_tokens(text, ratio=0.2):
    tokens = text.split()
    n_delete = int(len(tokens) * ratio)

    if n_delete == 0 or len(tokens) <= 1:
        return text

    delete_indices = set(random.sample(range(len(tokens)), n_delete))
    kept_tokens = [t for i, t in enumerate(tokens) if i not in delete_indices]
    return " ".join(kept_tokens)

noisy_queries = [delete_random_tokens(chunk, deletion_ratio) for chunk in sampled_chunks]

print("\n--- Example corruption ---")
print(f"Original: {sampled_chunks[0][:200]}...")
print(f"Noisy:    {noisy_queries[0][:200]}...")

# ===================== BUILD BENCHMARK DATAFRAME =====================
robustness_benchmark_df = pd.DataFrame({
    "id": range(len(sample_indices)),
    "original_chunk": sampled_chunks,
    "noisy_query": noisy_queries,
    "expected_file": [m["file_name"] for m in sampled_metadata],
    "expected_chunk_id": [m["chunk_id"] for m in sampled_metadata],
    "original_chunk_index": sample_indices
})

print(f"\nBuilt robustness benchmark with {len(robustness_benchmark_df)} queries")
print(robustness_benchmark_df[["noisy_query", "expected_file"]].head(3))

robustness_benchmark_df.to_csv("/content/drive/MyDrive/robustness_benchmark_20.csv", index=False)
print("\nSaved to robustness_benchmark_20.csv")

Found 100 embedding files


Loading embeddings:   0%|          | 0/100 [00:00<?, ?it/s]


============== LOADED DATA (jina-v5-nano) ==============
  Total books:      100
  Total chunks:     88275
  Embedding dim:    768
  Matrix size:      258.62 MB

Sampled 1000 chunks out of 88275 total

--- Example corruption ---
Original: پیامبری که وجه و خلیفه تام خداوند بر روی زمین بوده و واسطه و رحمت و فیض عوالم وجوی است بهفردی جز کسی که شایستگی این خطابات را داشته باشد، این کلام را نمی فرماید. عبارت دیگر قد اذنت لک است که اشاره دار...
Noisy:    که وجه و تام خداوند بر روی زمین بوده واسطه و و فیض عوالم وجوی است جز کسی که شایستگی این خطابات را داشته باشد، این کلام نمی فرماید. عبارت قد اذنت لک دارد به این که قبل از این این شما داده ام. (10) ... ...

Built robustness benchmark with 1000 queries
                                         noisy_query  \
0  که وجه و تام خداوند بر روی زمین بوده واسطه و و...   
1  دست از پاری او کشیدید و را که با اصرار و مکاتب...   
2  وقتی تو پایان رسید، حالت هیجان در مردم پدیدار ...   

                                       expected_file  
0  مرج البحرين ف

# Load model for query encoding

In [6]:
import torch
from sentence_transformers import SentenceTransformer

model_name = "jinaai/jina-embeddings-v5-text-nano-retrieval"
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = SentenceTransformer(model_name, trust_remote_code=True, device=device)
if torch.cuda.is_available():
    model = model.half()
print("Model loaded for query encoding!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/274 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/9.22k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.36k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/487 [00:00<?, ?B/s]

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}


tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/312 [00:00<?, ?B/s]

Model loaded for query encoding!


# Build Qdrant + Weaviate indexes for jina-v5-nano

In [7]:
!pip install -q qdrant-client weaviate-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 652.7/652.7 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 117.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 4.4 MB/s eta 0:00:00


In [16]:
import weaviate
from weaviate.classes.config import Configure, Property, DataType

# ===================== RECREATE WEAVIATE COLLECTION WITH chunk_id =====================
weaviate_collection = "BooksJina5nano"

if client_weaviate.collections.exists(weaviate_collection):
    client_weaviate.collections.delete(weaviate_collection)
    print(f"Deleted existing collection: {weaviate_collection}")

client_weaviate.collections.create(
    name=weaviate_collection,
    vectorizer_config=Configure.Vectorizer.none(),
    properties=[
        Property(name="file_name", data_type=DataType.TEXT),
        Property(name="chunk_id",  data_type=DataType.INT)
    ]
)
print(f"Created collection: {weaviate_collection} with chunk_id property\n")

# ===================== REINDEX =====================
print("Re-indexing into Weaviate with chunk_id...")
weaviate_index_start = time.time()

col = client_weaviate.collections.get(weaviate_collection)

with col.batch.fixed_size(batch_size=512) as batch:
    for i in tqdm(range(len(all_embeddings)), desc="Weaviate re-upload"):
        batch.add_object(
            properties={
                "file_name": all_metadata[i]["file_name"],
                "chunk_id":  all_metadata[i]["chunk_id"]
            },
            vector=all_embeddings[i].tolist()
        )

weaviate_index_time = time.time() - weaviate_index_start
print(f"Weaviate re-indexing time: {weaviate_index_time:.2f}s\n")

# Verify
print(f"Total objects: {col.aggregate.over_all(total_count=True).total_count}")

INFO:weaviate-client:Embedded weaviate wasn't listening on ports http:8079 & grpc:50050, so starting embedded weaviate again
INFO:weaviate-client:Started /root/.cache/weaviate-embedded: process ID 13666


Deleted existing collection: BooksJina5nano
Created collection: BooksJina5nano with chunk_id property

Re-indexing into Weaviate with chunk_id...


Weaviate re-upload:   0%|          | 0/88275 [00:00<?, ?it/s]

INFO:weaviate-client:Batch objects received exception: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNAVAILABLE
	details = "failed to connect to all addresses; last error: UNKNOWN: ipv6:%5B::1%5D:50050: Failed to connect to remote host: connect: Connection refused (111)"
	debug_error_string = "UNKNOWN:Error received from peer  {grpc_message:"failed to connect to all addresses; last error: UNKNOWN: ipv6:%5B::1%5D:50050: Failed to connect to remote host: connect: Connection refused (111)", grpc_status:14}"
>. Retrying with exponential backoff in 1 seconds
INFO:weaviate-client:Batch objects received exception: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNAVAILABLE
	details = "failed to connect to all addresses; last error: UNKNOWN: ipv6:%5B::1%5D:50050: Failed to connect to remote host: connect: Connection refused (111)"
	debug_error_string = "UNKNOWN:Error received from peer  {grpc_message:"failed to connect to all addresses; last error: 

KeyboardInterrupt: 

In [15]:
from qdrant_client.models import VectorParams, Distance, PointStruct

qdrant_collection = "books_jina5nano"

client_qdrant.delete_collection(collection_name=qdrant_collection)
client_qdrant.create_collection(
    collection_name=qdrant_collection,
    vectors_config=VectorParams(size=dim, distance=Distance.COSINE)
)

print("Re-indexing into Qdrant with chunk_id...")
qdrant_index_start = time.time()

for i in tqdm(range(0, len(all_embeddings), 512), desc="Qdrant re-upload"):
    batch_end = min(i + 512, len(all_embeddings))
    points = [
        PointStruct(
            id=j,
            vector=all_embeddings[j].tolist(),
            payload={
                "file_name": all_metadata[j]["file_name"],
                "chunk_id": all_metadata[j]["chunk_id"]
            }
        )
        for j in range(i, batch_end)
    ]
    client_qdrant.upsert(collection_name=qdrant_collection, points=points)

qdrant_index_time = time.time() - qdrant_index_start
print(f"Qdrant re-indexing time: {qdrant_index_time:.2f}s\n")

Re-indexing into Qdrant with chunk_id...


Qdrant re-upload:   0%|          | 0/173 [00:00<?, ?it/s]

Qdrant re-indexing time: 67.62s



# Evaluation

In [13]:
# ===================== UPDATED SEARCH FUNCTIONS (return chunk-level info) =====================

def qdrant_search_detailed(query_emb, k=50):
    """Returns raw chunk-level results: list of (file_name, chunk_id) tuples in rank order"""
    results = client_qdrant.query_points(
        collection_name=qdrant_collection,
        query=query_emb,
        limit=k,
        with_payload=True
    ).points

    return [(r.payload["file_name"], r.payload.get("chunk_id")) for r in results]


def weaviate_search_detailed(query_emb, k=50):
    """Returns raw chunk-level results: list of (file_name, chunk_id) tuples in rank order"""
    results = col.query.near_vector(
        near_vector=query_emb,
        limit=k,
        return_properties=["file_name", "chunk_id"]
    )

    return [(r.properties["file_name"], r.properties.get("chunk_id")) for r in results.objects]

In [8]:
def evaluate_robustness_full(search_fn_detailed, name):
    # Doc-level
    doc_hits_at_1, doc_hits_at_5, doc_hits_at_10 = 0, 0, 0
    # Chunk-level
    chunk_hits_at_1, chunk_hits_at_5, chunk_hits_at_10 = 0, 0, 0

    latencies = []

    for _, row in tqdm(robustness_benchmark_df.iterrows(), desc=f"Evaluating {name}", total=len(robustness_benchmark_df)):
        query = row["noisy_query"]
        expected_file  = row["expected_file"]
        expected_chunk = row["expected_chunk_id"]

        query_emb = model.encode([query], normalize_embeddings=True)[0].tolist()

        t0 = time.time()
        raw_results = search_fn_detailed(query_emb, k=50)   # [(file_name, chunk_id), ...] in rank order
        latency_ms = (time.time() - t0) * 1000
        latencies.append(latency_ms)

        # ---------- CHUNK-LEVEL: exact (file_name, chunk_id) match, no dedup ----------
        chunk_matches = [
            (fname == expected_file and cid == expected_chunk)
            for fname, cid in raw_results
        ]
        if any(chunk_matches[:1]):
            chunk_hits_at_1 += 1
        if any(chunk_matches[:5]):
            chunk_hits_at_5 += 1
        if any(chunk_matches[:10]):
            chunk_hits_at_10 += 1

        # ---------- DOC-LEVEL: deduplicate to unique files first ----------
        retrieved_files, seen = [], set()
        for fname, _ in raw_results:
            if fname not in seen:
                seen.add(fname)
                retrieved_files.append(fname)
            if len(retrieved_files) == 10:
                break

        if retrieved_files and retrieved_files[0] == expected_file:
            doc_hits_at_1 += 1
        if expected_file in retrieved_files[:5]:
            doc_hits_at_5 += 1
        if expected_file in retrieved_files[:10]:
            doc_hits_at_10 += 1

    n = len(robustness_benchmark_df)
    lat = np.array(latencies)

    return {
        "vector_store": name,
        "model": "jina-v5-nano",
        "test_type": f"{int(deletion_ratio*100)}%_token_deletion",

        # Document-level
        "doc_hit@1":  round(doc_hits_at_1 / n, 4),
        "doc_hit@5":  round(doc_hits_at_5 / n, 4),
        "doc_hit@10": round(doc_hits_at_10 / n, 4),

        # Chunk-level
        "chunk_hit@1":  round(chunk_hits_at_1 / n, 4),
        "chunk_hit@5":  round(chunk_hits_at_5 / n, 4),
        "chunk_hit@10": round(chunk_hits_at_10 / n, 4),

        "latency_mean_ms": round(lat.mean(), 2),
        "latency_p95_ms":  round(np.percentile(lat, 95), 2),
    }

# %20

In [16]:
qdrant_robustness_full   = evaluate_robustness_full(qdrant_search_detailed, "Qdrant")
weaviate_robustness_full = evaluate_robustness_full(weaviate_search_detailed, "Weaviate")

robustness_full_df = pd.DataFrame([qdrant_robustness_full, weaviate_robustness_full])
print(robustness_full_df.to_string(index=False))

robustness_full_df.to_csv("/content/drive/MyDrive/robustness_results_jina5nano_full.csv", index=False)

vector_store        model          test_type  doc_hit@1  doc_hit@5  doc_hit@10  chunk_hit@1  chunk_hit@5  chunk_hit@10  latency_mean_ms  latency_p95_ms
      Qdrant jina-v5-nano 20%_token_deletion      0.860      0.973       0.978        0.841        0.956         0.956           400.73          486.67
    Weaviate jina-v5-nano 20%_token_deletion      0.858      0.970       0.975        0.836        0.952         0.952             9.49           12.56


# %30

In [6]:
import os
import pickle
import time
import random
import numpy as np
import pandas as pd
import torch
from tqdm.notebook import tqdm
from sentence_transformers import SentenceTransformer

# ===================== CONFIG =====================
save_folder = "/content/drive/MyDrive/Book_Embeddings_100/jinaa5_nano"
model_name  = "jinaai/jina-embeddings-v5-text-nano-retrieval"

# ===================== LOAD MODEL =====================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Loading model on {device}...")

model = SentenceTransformer(model_name, trust_remote_code=True, device=device)
if torch.cuda.is_available():
    model = model.half()
print("Model loaded!\n")

# ===================== RELOAD ALL SAVED EMBEDDINGS =====================
all_chunks_text = []
all_embeddings  = []
all_metadata    = []

pkl_files = [f for f in os.listdir(save_folder) if f.endswith('.pkl')]
print(f"Found {len(pkl_files)} embedding files")

for pkl_file in tqdm(pkl_files, desc="Loading embeddings"):
    file_path = os.path.join(save_folder, pkl_file)
    with open(file_path, "rb") as f:
        data = pickle.load(f)

    chunks     = data["chunks"]
    embeddings = data["embeddings"]
    book_title = data["book_title"]
    file_name  = data["file_name"]

    for i, (chunk, emb) in enumerate(zip(chunks, embeddings)):
        all_chunks_text.append(chunk)
        all_embeddings.append(emb)
        all_metadata.append({
            "file_name": file_name,
            "book_title": book_title,
            "chunk_id": i
        })

all_embeddings = np.array(all_embeddings).astype('float32')

print(f"""
============== LOADED DATA (jina-v5-nano) ==============
  Total books:      {len(pkl_files)}
  Total chunks:     {len(all_chunks_text)}
  Embedding dim:    {all_embeddings.shape[1]}
  Matrix size:      {all_embeddings.nbytes/1024**2:.2f} MB
=========================================================
""")

Loading model on cuda...


[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Loading weights:   0%|          | 0/110 [00:00<?, ?it/s]

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}


Model loaded!

Found 100 embedding files


Loading embeddings:   0%|          | 0/100 [00:00<?, ?it/s]


============== LOADED DATA (jina-v5-nano) ==============
  Total books:      100
  Total chunks:     88275
  Embedding dim:    768
  Matrix size:      258.62 MB



In [9]:
!pip install -q qdrant-client weaviate-client

from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
import weaviate
from weaviate.classes.config import Configure, Property, DataType

dim = all_embeddings.shape[1]

# ---- Qdrant ----
client_qdrant = QdrantClient(":memory:")
qdrant_collection = "books_jina5nano"

client_qdrant.create_collection(
    collection_name=qdrant_collection,
    vectors_config=VectorParams(size=dim, distance=Distance.COSINE)
)

print("Re-indexing into Qdrant...")
qdrant_index_start = time.time()
for i in tqdm(range(0, len(all_embeddings), 512), desc="Qdrant upload"):
    batch_end = min(i + 512, len(all_embeddings))
    points = [
        PointStruct(
            id=j,
            vector=all_embeddings[j].tolist(),
            payload={
                "file_name": all_metadata[j]["file_name"],
                "chunk_id": all_metadata[j]["chunk_id"]
            }
        )
        for j in range(i, batch_end)
    ]
    client_qdrant.upsert(collection_name=qdrant_collection, points=points)
qdrant_index_time = time.time() - qdrant_index_start
print(f"Qdrant indexing time: {qdrant_index_time:.2f}s\n")

# ---- Weaviate ----
client_weaviate = weaviate.connect_to_embedded()
weaviate_collection = "BooksJina5nano"

if client_weaviate.collections.exists(weaviate_collection):
    client_weaviate.collections.delete(weaviate_collection)

client_weaviate.collections.create(
    name=weaviate_collection,
    vectorizer_config=Configure.Vectorizer.none(),
    properties=[
        Property(name="file_name", data_type=DataType.TEXT),
        Property(name="chunk_id",  data_type=DataType.INT)
    ]
)

print("Re-indexing into Weaviate...")
weaviate_index_start = time.time()
col = client_weaviate.collections.get(weaviate_collection)
with col.batch.fixed_size(batch_size=512) as batch:
    for i in tqdm(range(len(all_embeddings)), desc="Weaviate upload"):
        batch.add_object(
            properties={
                "file_name": all_metadata[i]["file_name"],
                "chunk_id":  all_metadata[i]["chunk_id"]
            },
            vector=all_embeddings[i].tolist()
        )
weaviate_index_time = time.time() - weaviate_index_start
print(f"Weaviate indexing time: {weaviate_index_time:.2f}s\n")

# ---- Search functions ----
def qdrant_search_detailed(query_emb, k=50):
    results = client_qdrant.query_points(
        collection_name=qdrant_collection, query=query_emb, limit=k, with_payload=True
    ).points
    return [(r.payload["file_name"], r.payload.get("chunk_id")) for r in results]

def weaviate_search_detailed(query_emb, k=50):
    results = col.query.near_vector(near_vector=query_emb, limit=k, return_properties=["file_name", "chunk_id"])
    return [(r.properties["file_name"], r.properties.get("chunk_id")) for r in results.objects]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 652.7/652.7 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 96.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 3.0 MB/s eta 0:00:00
  Attempting uninstall: grpcio
    Found existing installation: grpcio 1.81.1
    Uninstalling grpcio-1.81.1:
      Successfully uninstalled grpcio-1.81.1
Re-indexing into Qdrant...


Qdrant upload:   0%|          | 0/173 [00:00<?, ?it/s]

/tmp/ipykernel_501/4215261234.py:34: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 20480 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client_qdrant.upsert(collection_name=qdrant_collection, points=points)
INFO:weaviate-client:Binary /root/.cache/weaviate-embedded did not exist. Downloading binary from https://github.com/weaviate/weaviate/releases/download/v1.30.5/weaviate-v1.30.5-Linux-amd64.tar.gz


Qdrant indexing time: 67.14s



INFO:weaviate-client:Started /root/.cache/weaviate-embedded: process ID 6279


Re-indexing into Weaviate...


Weaviate upload:   0%|          | 0/88275 [00:00<?, ?it/s]

Weaviate indexing time: 129.43s



# 30%

In [7]:
import random
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

# ===================== CONFIG =====================
n_samples        = 1000
deletion_ratio   = 0.30   # ← changed from 0.20
random_seed      = 42

random.seed(random_seed)
np.random.seed(random_seed)

# ===================== SAMPLE 1000 RANDOM CHUNKS (same seed = same chunks as before) =====================
total_chunks = len(all_chunks_text)
sample_indices = random.sample(range(total_chunks), min(n_samples, total_chunks))

print(f"Sampled {len(sample_indices)} chunks out of {total_chunks} total")

sampled_chunks     = [all_chunks_text[i] for i in sample_indices]
sampled_metadata   = [all_metadata[i] for i in sample_indices]
sampled_embeddings = all_embeddings[sample_indices]

# ===================== DELETE 30% OF TOKENS RANDOMLY =====================
def delete_random_tokens(text, ratio=0.3):
    tokens = text.split()
    n_delete = int(len(tokens) * ratio)

    if n_delete == 0 or len(tokens) <= 1:
        return text

    delete_indices = set(random.sample(range(len(tokens)), n_delete))
    kept_tokens = [t for i, t in enumerate(tokens) if i not in delete_indices]
    return " ".join(kept_tokens)

noisy_queries = [delete_random_tokens(chunk, deletion_ratio) for chunk in sampled_chunks]

print("\n--- Example corruption (30%) ---")
print(f"Original: {sampled_chunks[0][:200]}...")
print(f"Noisy:    {noisy_queries[0][:200]}...")

# ===================== BUILD BENCHMARK DATAFRAME =====================
robustness_benchmark_df_30 = pd.DataFrame({
    "id": range(len(sample_indices)),
    "original_chunk": sampled_chunks,
    "noisy_query": noisy_queries,
    "expected_file": [m["file_name"] for m in sampled_metadata],
    "expected_chunk_id": [m["chunk_id"] for m in sampled_metadata],
    "original_chunk_index": sample_indices
})

print(f"\nBuilt 30% robustness benchmark with {len(robustness_benchmark_df_30)} queries")

robustness_benchmark_df_30.to_csv("/content/drive/MyDrive/robustness_benchmark_1000_jina5nano_30pct.csv", index=False)
print("Saved to robustness_benchmark_1000_jina5nano_30pct.csv")

Sampled 1000 chunks out of 88275 total

--- Example corruption (30%) ---
Original: پیامبری که وجه و خلیفه تام خداوند بر روی زمین بوده و واسطه و رحمت و فیض عوالم وجوی است بهفردی جز کسی که شایستگی این خطابات را داشته باشد، این کلام را نمی فرماید. عبارت دیگر قد اذنت لک است که اشاره دار...
Noisy:    که وجه و تام بر روی زمین بوده واسطه و و فیض عوالم وجوی است جز کسی که این خطابات را داشته باشد، این کلام نمی فرماید. عبارت قد اذنت لک دارد به این قبل از این این شما داده ام. ... نحو الكساء و السلام أبت...

Built 30% robustness benchmark with 1000 queries
Saved to robustness_benchmark_1000_jina5nano_30pct.csv


In [10]:
def evaluate_robustness_full(search_fn_detailed, name, benchmark_df, deletion_pct):
    doc_hits_at_1, doc_hits_at_5, doc_hits_at_10 = 0, 0, 0
    chunk_hits_at_1, chunk_hits_at_5, chunk_hits_at_10 = 0, 0, 0
    latencies = []

    for _, row in tqdm(benchmark_df.iterrows(), desc=f"Evaluating {name} ({deletion_pct}%)", total=len(benchmark_df)):
        query = row["noisy_query"]
        expected_file  = row["expected_file"]
        expected_chunk = row["expected_chunk_id"]

        query_emb = model.encode([query], normalize_embeddings=True)[0].tolist()

        t0 = time.time()
        raw_results = search_fn_detailed(query_emb, k=50)
        latency_ms = (time.time() - t0) * 1000
        latencies.append(latency_ms)

        # Chunk-level
        chunk_matches = [
            (fname == expected_file and cid == expected_chunk)
            for fname, cid in raw_results
        ]
        if any(chunk_matches[:1]):
            chunk_hits_at_1 += 1
        if any(chunk_matches[:5]):
            chunk_hits_at_5 += 1
        if any(chunk_matches[:10]):
            chunk_hits_at_10 += 1

        # Doc-level
        retrieved_files, seen = [], set()
        for fname, _ in raw_results:
            if fname not in seen:
                seen.add(fname)
                retrieved_files.append(fname)
            if len(retrieved_files) == 10:
                break

        if retrieved_files and retrieved_files[0] == expected_file:
            doc_hits_at_1 += 1
        if expected_file in retrieved_files[:5]:
            doc_hits_at_5 += 1
        if expected_file in retrieved_files[:10]:
            doc_hits_at_10 += 1

    n = len(benchmark_df)
    lat = np.array(latencies)

    return {
        "vector_store": name,
        "model": "jina-v5-nano",
        "test_type": f"{deletion_pct}%_token_deletion",
        "doc_hit@1":  round(doc_hits_at_1 / n, 4),
        "doc_hit@5":  round(doc_hits_at_5 / n, 4),
        "doc_hit@10": round(doc_hits_at_10 / n, 4),
        "chunk_hit@1":  round(chunk_hits_at_1 / n, 4),
        "chunk_hit@5":  round(chunk_hits_at_5 / n, 4),
        "chunk_hit@10": round(chunk_hits_at_10 / n, 4),
        "latency_mean_ms": round(lat.mean(), 2),
        "latency_p95_ms":  round(np.percentile(lat, 95), 2),
    }

# ===================== RUN (30% deletion) =====================
qdrant_robustness_30   = evaluate_robustness_full(qdrant_search_detailed, "Qdrant", robustness_benchmark_df_30, 30)
weaviate_robustness_30 = evaluate_robustness_full(weaviate_search_detailed, "Weaviate", robustness_benchmark_df_30, 30)

robustness_30_df = pd.DataFrame([qdrant_robustness_30, weaviate_robustness_30])
print(robustness_30_df.to_string(index=False))

robustness_30_df.to_csv("/content/drive/MyDrive/robustness_results_jina5nano_30pct.csv", index=False)

Evaluating Qdrant (30%):   0%|          | 0/1000 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Evaluating Weaviate (30%):   0%|          | 0/1000 [00:00<?, ?it/s]

vector_store        model          test_type  doc_hit@1  doc_hit@5  doc_hit@10  chunk_hit@1  chunk_hit@5  chunk_hit@10  latency_mean_ms  latency_p95_ms
      Qdrant jina-v5-nano 30%_token_deletion      0.833      0.968       0.975        0.812        0.947         0.950           404.82          487.24
    Weaviate jina-v5-nano 30%_token_deletion      0.823      0.962       0.973        0.802        0.939         0.943             8.63           11.39


# %40

In [11]:
import random
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

# ===================== CONFIG =====================
n_samples        = 1000
deletion_ratio   = 0.40   # ← changed from 0.20
random_seed      = 42

random.seed(random_seed)
np.random.seed(random_seed)

# ===================== SAMPLE 1000 RANDOM CHUNKS (same seed = same chunks as before) =====================
total_chunks = len(all_chunks_text)
sample_indices = random.sample(range(total_chunks), min(n_samples, total_chunks))

print(f"Sampled {len(sample_indices)} chunks out of {total_chunks} total")

sampled_chunks     = [all_chunks_text[i] for i in sample_indices]
sampled_metadata   = [all_metadata[i] for i in sample_indices]
sampled_embeddings = all_embeddings[sample_indices]

# ===================== DELETE 40% OF TOKENS RANDOMLY =====================
def delete_random_tokens(text, ratio=0.4):
    tokens = text.split()
    n_delete = int(len(tokens) * ratio)

    if n_delete == 0 or len(tokens) <= 1:
        return text

    delete_indices = set(random.sample(range(len(tokens)), n_delete))
    kept_tokens = [t for i, t in enumerate(tokens) if i not in delete_indices]
    return " ".join(kept_tokens)

noisy_queries = [delete_random_tokens(chunk, deletion_ratio) for chunk in sampled_chunks]

print("\n--- Example corruption (40%) ---")
print(f"Original: {sampled_chunks[0][:200]}...")
print(f"Noisy:    {noisy_queries[0][:200]}...")

# ===================== BUILD BENCHMARK DATAFRAME =====================
robustness_benchmark_df_40 = pd.DataFrame({
    "id": range(len(sample_indices)),
    "original_chunk": sampled_chunks,
    "noisy_query": noisy_queries,
    "expected_file": [m["file_name"] for m in sampled_metadata],
    "expected_chunk_id": [m["chunk_id"] for m in sampled_metadata],
    "original_chunk_index": sample_indices
})

print(f"\nBuilt 40% robustness benchmark with {len(robustness_benchmark_df_40)} queries")

robustness_benchmark_df_40.to_csv("/content/drive/MyDrive/robustness_benchmark_1000_jina5nano_40pct.csv", index=False)
print("Saved to robustness_benchmark_1000_jina5nano_40pct.csv")

Sampled 1000 chunks out of 88275 total

--- Example corruption (40%) ---
Original: پیامبری که وجه و خلیفه تام خداوند بر روی زمین بوده و واسطه و رحمت و فیض عوالم وجوی است بهفردی جز کسی که شایستگی این خطابات را داشته باشد، این کلام را نمی فرماید. عبارت دیگر قد اذنت لک است که اشاره دار...
Noisy:    که وجه و تام روی زمین بوده واسطه و و عوالم وجوی است جز کسی که این را داشته باشد، این نمی عبارت قد اذنت لک دارد این قبل از این این شما داده ام. ... نحو الكساء و السلام أبتاه يا أ تأذن لي أكون معكم الكس...

Built 40% robustness benchmark with 1000 queries
Saved to robustness_benchmark_1000_jina5nano_40pct.csv


In [14]:
# ===================== RUN (40% deletion) =====================
qdrant_robustness_40   = evaluate_robustness_full(qdrant_search_detailed, "Qdrant", robustness_benchmark_df_40, 40)
weaviate_robustness_40 = evaluate_robustness_full(weaviate_search_detailed, "Weaviate", robustness_benchmark_df_40, 40)

robustness_40_df = pd.DataFrame([qdrant_robustness_40, weaviate_robustness_40])
print(robustness_40_df.to_string(index=False))

robustness_40_df.to_csv("/content/drive/MyDrive/robustness_results_jina5nano_40pct.csv", index=False)

Evaluating Qdrant (40%):   0%|          | 0/1000 [00:00<?, ?it/s]

Evaluating Weaviate (40%):   0%|          | 0/1000 [00:00<?, ?it/s]

INFO:weaviate-client:Searching in collection BooksJina5nano received exception: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNAVAILABLE
	details = "failed to connect to all addresses; last error: UNKNOWN: ipv4:127.0.0.1:50050: Failed to connect to remote host: connect: Connection refused (111)"
	debug_error_string = "UNKNOWN:Error received from peer  {grpc_status:14, grpc_message:"failed to connect to all addresses; last error: UNKNOWN: ipv4:127.0.0.1:50050: Failed to connect to remote host: connect: Connection refused (111)"}"
>. Retrying with exponential backoff in 1 seconds
INFO:weaviate-client:Searching in collection BooksJina5nano received exception: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNAVAILABLE
	details = "failed to connect to all addresses; last error: UNKNOWN: ipv6:%5B::1%5D:50050: Failed to connect to remote host: connect: Connection refused (111)"
	debug_error_string = "UNKNOWN:Error received from peer  {grpc_status:

WeaviateQueryError: Query call with protocol GRPC search failed with message The request to Weaviate failed after 5 retries. Details: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNAVAILABLE
	details = "failed to connect to all addresses; last error: UNKNOWN: ipv6:%5B::1%5D:50050: Failed to connect to remote host: connect: Connection refused (111)"
	debug_error_string = "UNKNOWN:Error received from peer  {grpc_status:14, grpc_message:"failed to connect to all addresses; last error: UNKNOWN: ipv6:%5B::1%5D:50050: Failed to connect to remote host: connect: Connection refused (111)"}"
>.

# %50

In [5]:
books_folder = "/content/drive/MyDrive/100_books/"

# Fixed chunk settings for fair comparison
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=90,
    separators=["\n\n", "\n", "۔", "؟", "!", ".", " ", ""]
)

htm_files = [f for f in os.listdir(books_folder) if f.lower().endswith(('.htm', '.html'))]
actions = []

# --- Tracking variables ---
total_chunks_created = 0
total_embedding_time = 0.0  # time spent only on model.encode()

for file_name in tqdm(htm_files):          # Start with 20 books
    base_name = file_name.replace(".htm", "").replace(".html", "")


    file_path = os.path.join(books_folder, file_name)
    text = extract_text_from_htm(file_path)

    # if len(text) < 300:
    #     continue

    # chunks = chunk_text(text)
    chunks = text_splitter.split_text(text)
    total_chunks_created += len(chunks)

100%|██████████| 100/100 [01:01<00:00,  1.62it/s]


# Create Random test queries from chunks

In [ ]:

# ===================== CONFIG =====================
n_samples        = 1000
deletion_ratio   = 0.40   #****&&&&&&&&*****@@@@@@@@@*******$$$$$$$$$***
random_seed      = 42

random.seed(random_seed)
np.random.seed(random_seed)

# ===================== SAMPLE 1000 RANDOM CHUNKS (same seed = same chunks as before) =====================
total_chunks = len(all_chunks_text)
sample_indices = random.sample(range(total_chunks), min(n_samples, total_chunks))

print(f"Sampled {len(sample_indices)} chunks out of {total_chunks} total")

sampled_chunks     = [all_chunks_text[i] for i in sample_indices]
sampled_metadata   = [all_metadata[i] for i in sample_indices]
sampled_embeddings = all_embeddings[sample_indices]

# ===================== DELETE n% OF TOKENS RANDOMLY =====================

def delete_random_tokens(text, ratio=0.4):  #****&&&&&&&&*****@@@@@@@@@*******$$$$$$$$$***
    tokens = text.split()
    n_delete = int(len(tokens) * ratio)

    if n_delete == 0 or len(tokens) <= 1:
        return text

    delete_indices = set(random.sample(range(len(tokens)), n_delete))
    kept_tokens = [t for i, t in enumerate(tokens) if i not in delete_indices]
    return " ".join(kept_tokens)

noisy_queries = [delete_random_tokens(chunk, deletion_ratio) for chunk in sampled_chunks]

print("\n--- Example corruption (n%) ---")
print(f"Original: {sampled_chunks[0][:200]}...")
print(f"Noisy:    {noisy_queries[0][:200]}...")

# ===================== BUILD BENCHMARK DATAFRAME =====================

#****&&&&&&&&*****@@@@@@@@@*******$$$$$$$$$***
robustness_benchmark_df_40 = pd.DataFrame({
    "id": range(len(sample_indices)),
    "original_chunk": sampled_chunks,
    "noisy_query": noisy_queries,
    "expected_file": [m["file_name"] for m in sampled_metadata],
    "expected_chunk_id": [m["chunk_id"] for m in sampled_metadata],
    "original_chunk_index": sample_indices
})

#****&&&&&&&&*****@@@@@@@@@*******$$$$$$$$$***
print(f"\nBuilt robustness benchmark with {len(robustness_benchmark_df_40)} queries")

robustness_benchmark_df_30.to_csv("/content/drive/MyDrive/robustness_benchmark_1000_jina5nano_40pct.csv", index=False)
print("Saved to robustness_benchmark_1000_jina5nano_40pct.csv")